# LeetCode 1868 — Product of Two Run-Length Encoded Arrays bằng Python

Link đề: https://leetcode.com/problems/product-of-two-run-length-encoded-arrays/ · Độ khó: **Medium** · Chủ đề: Array, Two Pointers

> **Lưu ý: đây là bài Premium** trên LeetCode — cần tài khoản Premium để xem đề gốc và nộp bài. Đề dưới đây viết theo hiểu biết về bài (nguồn: bản đề của doocs/leetcode); nếu có tài khoản Premium, hãy **đối chiếu lại đề gốc**.

Skeleton LeetCode:

```python
class Solution:
    def findRLEArray(self, encoded1: List[List[int]], encoded2: List[List[int]]) -> List[List[int]]:
        ...
```

## 1. Mục tiêu học

1. Hiểu **mã hóa độ dài loạt** (run-length encoding, RLE): mảng `[[giá trị, tần suất], …]` biểu diễn gọn mảng có nhiều đoạn lặp.
2. Làm phép toán **từng phần tử** trên hai mảng **mà không giải nén**: duyệt hai danh sách đoạn bằng **hai con trỏ**, mỗi bước "tiêu thụ" `min(tần suất còn lại)`.
3. Đảm bảo kết quả có **độ dài nhỏ nhất**: gộp đoạn mới vào đoạn cuối nếu cùng giá trị.
4. Phân tích độ phức tạp theo **số đoạn** (`≤ 2·10⁵`) thay vì độ dài mảng thật (tới `10⁹`).
5. Viết brute force "giải nén – nhân – nén" để kiểm chứng trên dữ liệu nhỏ.

## 2. Đề bài & ví dụ

**RLE:** mảng `nums` có nhiều đoạn số lặp liên tiếp được biểu diễn bởi mảng 2 chiều `encoded`, mỗi `encoded[i] = [val_i, freq_i]` nghĩa là đoạn thứ `i` gồm `val_i` lặp `freq_i` lần. Ví dụ `nums = [1,1,1,2,2,2,2,2]` ⇔ `encoded = [[1,3],[2,5]]` ("ba số 1 rồi năm số 2").

**Tích** của hai mảng RLE `encoded1`, `encoded2`:
1. Giải nén thành `nums1`, `nums2` (cùng độ dài).
2. Tạo `prodNums[i] = nums1[i] * nums2[i]`.
3. Nén `prodNums` thành RLE và trả về.

Trả về tích của `encoded1` và `encoded2`. **Lưu ý:** kết quả nén phải có **độ dài nhỏ nhất có thể** (không có hai đoạn kề nhau cùng giá trị).

**Ví dụ 1:** `encoded1 = [[1,3],[2,3]]`, `encoded2 = [[6,3],[3,3]]` → `[[6,6]]` (`prodNums = [6,6,6,6,6,6]`).

**Ví dụ 2:** `encoded1 = [[1,3],[2,1],[3,2]]`, `encoded2 = [[2,3],[3,3]]` → `[[2,3],[6,1],[9,2]]` (`prodNums = [2,2,2,6,9,9]`).

**Ràng buộc:** `1 ≤ len(encoded1), len(encoded2) ≤ 10⁵`; `1 ≤ val, freq ≤ 10⁴`; hai mảng giải nén có cùng độ dài.

*(Đề Premium — đã diễn đạt lại, hãy đối chiếu lại đề gốc.)*

In [1]:
import random
import time
import bisect
from itertools import accumulate, groupby
from typing import List

examples = [
    (([[1, 3], [2, 3]], [[6, 3], [3, 3]]), [[6, 6]]),
    (([[1, 3], [2, 1], [3, 2]], [[2, 3], [3, 3]]), [[2, 3], [6, 1], [9, 2]]),
]
for (e1, e2), exp in examples:
    print(f"{e1} x {e2} -> kỳ vọng {exp}")

[[1, 3], [2, 3]] x [[6, 3], [3, 3]] -> kỳ vọng [[6, 6]]
[[1, 3], [2, 1], [3, 2]] x [[2, 3], [3, 3]] -> kỳ vọng [[2, 3], [6, 1], [9, 2]]


## 3. Kiến thức nền

### 3.1 Giải nén và nén RLE

In [2]:
def decode(enc):
    out = []
    for v, f in enc:
        out.extend([v] * f)
    return out

def encode(nums):
    return [[v, len(list(g))] for v, g in groupby(nums)]

e = [[1, 3], [2, 5]]
print(decode(e), "->", encode(decode(e)))

[1, 1, 1, 2, 2, 2, 2, 2] -> [[1, 3], [2, 5]]


`itertools.groupby` gom các phần tử **liên tiếp** bằng nhau — đúng định nghĩa nén RLE tối thiểu.

### 3.2 Độ dài thật có thể rất lớn

Mỗi mảng có tới `10⁵` đoạn, mỗi đoạn tới `10⁴` phần tử ⇒ độ dài thật tới `10⁹`. Giải nén là **không khả thi** về thời gian lẫn bộ nhớ (một list `10⁹` số Python cần hàng chục GB).

In [3]:
max_len = 10 ** 5 * 10 ** 4
print(f"Độ dài tối đa: {max_len:,} phần tử ≈ {max_len * 8 / 2 ** 30:.0f} GiB chỉ riêng con trỏ của list")

Độ dài tối đa: 1,000,000,000 phần tử ≈ 7 GiB chỉ riêng con trỏ của list


### 3.3 Hai con trỏ trên hai danh sách đoạn (giống merge)

Như bước trộn của merge sort, nhưng thay vì so sánh giá trị, ta so sánh **lượng còn lại** của đoạn hiện tại ở mỗi bên. Đoạn nào hết trước thì con trỏ bên đó tiến lên.

### 3.4 Gộp vào đoạn cuối

Khi sinh ra đoạn `(v, f)` mới: nếu kết quả rỗng hoặc đoạn cuối có giá trị khác `v` ⇒ thêm mới; ngược lại cộng `f` vào đoạn cuối. Đảm bảo kết quả **tối thiểu**.

In [4]:
def append_merge(res, v, f):
    if res and res[-1][0] == v:
        res[-1][1] += f
    else:
        res.append([v, f])

res = []
for v, f in [(6, 3), (6, 3), (2, 1), (2, 4), (6, 1)]:
    append_merge(res, v, f)
print(res)

[[6, 6], [2, 5], [6, 1]]


## 4. Ý tưởng nền tảng / trực giác

Trên trục vị trí `0..N−1`, mỗi mảng chia trục thành các **đoạn hằng**. Giao của hai cách chia là một cách chia mịn hơn: trên mỗi mảnh nhỏ, cả `nums1` và `nums2` đều hằng ⇒ tích cũng hằng. Số mảnh `≤ n₁ + n₂ − 1` (mỗi ranh giới của một trong hai mảng tạo tối đa một mảnh mới).

Tính tay **Ví dụ 2**:

| mảnh vị trí | đoạn của nums1 | đoạn của nums2 | tích | độ dài |
|---|---|---|---|---|
| 0..2 | `1 (×3)` | `2 (×3)` | 2 | 3 |
| 3 | `2 (×1)` | `3 (×3)` – còn 3 | 6 | 1 |
| 4..5 | `3 (×2)` | `3` – còn 2 | 9 | 2 |

Không có hai mảnh kề cùng tích ⇒ kết quả `[[2,3],[6,1],[9,2]]`. Ở **Ví dụ 1**, hai mảnh `1·6 = 6` và `2·3 = 6` cùng giá trị ⇒ gộp thành `[[6,6]]`.

In [5]:
(e1, e2), _ = examples[1]
b1 = list(accumulate(f for _, f in e1)); b2 = list(accumulate(f for _, f in e2))
print("ranh giới (vị trí kết thúc, tính từ 1) nums1:", b1, "| nums2:", b2, "| hợp:", sorted(set(b1) | set(b2)))

ranh giới (vị trí kết thúc, tính từ 1) nums1: [3, 4, 6] | nums2: [3, 6] | hợp: [3, 4, 6]


## 5. Cách 1 — Brute Force / baseline: giải nén – nhân – nén

In [6]:
def rle_product_bruteforce(encoded1, encoded2):
    a, b = decode(encoded1), decode(encoded2)
    assert len(a) == len(b)
    return encode([x * y for x, y in zip(a, b)])

for (e1, e2), exp in examples:
    got = rle_product_bruteforce(e1, e2); print(got, "kỳ vọng", exp); assert got == exp

[[6, 6]] kỳ vọng [[6, 6]]
[[2, 3], [6, 1], [9, 2]] kỳ vọng [[2, 3], [6, 1], [9, 2]]


**Độ phức tạp:** `O(N)` thời gian và bộ nhớ với `N = Σ freq` tới `10⁹` ⇒ **TLE/MLE**. Chỉ dùng kiểm chứng.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (tiêu thụ theo đoạn):** giữ con trỏ `i` (đoạn của `encoded1`), `j` (đoạn của `encoded2`) và lượng **còn lại** `r1`, `r2` của đoạn hiện tại. Mỗi bước: `f = min(r1, r2)`, sinh mảnh `(v1·v2, f)`, trừ `f` khỏi cả hai; đoạn nào về 0 thì tiến con trỏ bên đó (có thể cả hai cùng lúc).

**Insight 2 (gộp):** sinh mảnh xong thì `append_merge` vào kết quả ⇒ tối thiểu.

**Insight 3 (không sửa input):** dùng biến `r1`, `r2` thay vì trừ thẳng vào `encoded1[i][1]` — tránh hiệu ứng phụ trên dữ liệu người gọi.

**Invariant:** sau mỗi bước, kết quả là RLE tối thiểu của `prodNums[0 .. p)` với `p` = tổng các `f` đã sinh; `r1` = số phần tử chưa dùng của đoạn `i`, tương tự `r2`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Hai con trỏ + gộp (bản nộp)

In [7]:
def rle_product_two_pointers(encoded1, encoded2):
    res = []
    i = j = 0
    r1, r2 = encoded1[0][1], encoded2[0][1]
    while i < len(encoded1):
        f = min(r1, r2)
        append_merge(res, encoded1[i][0] * encoded2[j][0], f)
        r1 -= f; r2 -= f
        if r1 == 0:
            i += 1
            if i < len(encoded1):
                r1 = encoded1[i][1]
        if r2 == 0:
            j += 1
            if j < len(encoded2):
                r2 = encoded2[j][1]
    return res

print([rle_product_two_pointers(*a) for a, _ in examples])

[[[6, 6]], [[2, 3], [6, 1], [9, 2]]]


Vì hai mảng có **cùng độ dài**, `encoded1` và `encoded2` hết **cùng lúc** ⇒ điều kiện `while i < len(encoded1)` là đủ (khi `i` hết thì `j` cũng hết).

**Độ phức tạp Cách 2:** mỗi bước ít nhất một con trỏ tiến ⇒ `≤ n₁ + n₂` bước ⇒ `O(n₁ + n₂)` thời gian, `O(1)` phụ trợ ngoài kết quả.

### 7.2 Cách 3 — Hợp các ranh giới + `bisect`

Tính tổng tiền tố tần suất để có các **vị trí kết thúc** của đoạn ở mỗi mảng; hợp hai tập ranh giới (đã sắp). Với mỗi mảnh `[trước, sau)`, tìm đoạn chứa nó trong mỗi mảng bằng `bisect`.

In [8]:
def rle_product_boundaries(encoded1, encoded2):
    p1 = list(accumulate(f for _, f in encoded1))
    p2 = list(accumulate(f for _, f in encoded2))
    cuts = sorted(set(p1) | set(p2))
    res, prev = [], 0
    for c in cuts:
        i = bisect.bisect_left(p1, c)          # đoạn đầu tiên kết thúc tại hoặc sau c
        j = bisect.bisect_left(p2, c)
        append_merge(res, encoded1[i][0] * encoded2[j][0], c - prev)
        prev = c
    return res

print([rle_product_boundaries(*a) for a, _ in examples])

[[[6, 6]], [[2, 3], [6, 1], [9, 2]]]


Mảnh `[prev, c)` nằm trọn trong đoạn đầu tiên có vị trí kết thúc `≥ c`, vì `c` là một ranh giới và không có ranh giới nào nằm giữa `prev` và `c`.

**Độ phức tạp Cách 3:** `O((n₁ + n₂) log(n₁ + n₂))`.

### 7.3 Cách 4 — Trộn ranh giới tuyến tính (không cần `bisect`)

Cải tiến Cách 3: vì cả `p1`, `p2` đã tăng, trộn chúng như merge và theo dõi chỉ số đoạn hiện tại — tương đương Cách 2 nhưng nhìn theo **vị trí tuyệt đối**.

In [9]:
class Solution:
    def findRLEArray(self, encoded1: List[List[int]], encoded2: List[List[int]]) -> List[List[int]]:
        res = []
        i = j = 0
        end1, end2 = encoded1[0][1], encoded2[0][1]      # vị trí kết thúc (tuyệt đối) của đoạn hiện tại
        pos = 0
        while i < len(encoded1):
            nxt = min(end1, end2)
            v = encoded1[i][0] * encoded2[j][0]
            if res and res[-1][0] == v:
                res[-1][1] += nxt - pos
            else:
                res.append([v, nxt - pos])
            pos = nxt
            if end1 == pos:
                i += 1
                if i < len(encoded1):
                    end1 += encoded1[i][1]
            if end2 == pos:
                j += 1
                if j < len(encoded2):
                    end2 += encoded2[j][1]
        return res

print([Solution().findRLEArray(*a) for a, _ in examples])

[[[6, 6]], [[2, 3], [6, 1], [9, 2]]]


**Độ phức tạp Cách 4:** `O(n₁ + n₂)`.

## 8. TRACE từng bước

Trace Cách 2 trên **Ví dụ 2**:

In [10]:
def trace_two_pointers(encoded1, encoded2):
    res = []; i = j = 0; r1, r2 = encoded1[0][1], encoded2[0][1]
    while i < len(encoded1):
        f = min(r1, r2); v = encoded1[i][0] * encoded2[j][0]
        before = [x[:] for x in res]
        append_merge(res, v, f)
        act = "gộp" if len(res) == len(before) else "thêm"
        print(f"i={i} (v1={encoded1[i][0]}, còn {r1}) | j={j} (v2={encoded2[j][0]}, còn {r2}) -> mảnh ({v}, {f}) {act:<4} res={res}")
        r1 -= f; r2 -= f
        if r1 == 0:
            i += 1; r1 = encoded1[i][1] if i < len(encoded1) else 0
        if r2 == 0:
            j += 1; r2 = encoded2[j][1] if j < len(encoded2) else 0
    return res

assert trace_two_pointers(*examples[1][0]) == examples[1][1]
print("---")
assert trace_two_pointers(*examples[0][0]) == examples[0][1]

i=0 (v1=1, còn 3) | j=0 (v2=2, còn 3) -> mảnh (2, 3) thêm res=[[2, 3]]
i=1 (v1=2, còn 1) | j=1 (v2=3, còn 3) -> mảnh (6, 1) thêm res=[[2, 3], [6, 1]]
i=2 (v1=3, còn 2) | j=1 (v2=3, còn 2) -> mảnh (9, 2) thêm res=[[2, 3], [6, 1], [9, 2]]
---
i=0 (v1=1, còn 3) | j=0 (v2=6, còn 3) -> mảnh (6, 3) thêm res=[[6, 3]]
i=1 (v1=2, còn 3) | j=1 (v2=3, còn 3) -> mảnh (6, 3) gộp  res=[[6, 6]]


| bước | đoạn 1 (còn) | đoạn 2 (còn) | f | mảnh | hành động | res |
|---|---|---|---|---|---|---|
| 1 | `1` (3) | `2` (3) | 3 | (2, 3) | thêm | `[[2,3]]` |
| 2 | `2` (1) | `3` (3) | 1 | (6, 1) | thêm | `[[2,3],[6,1]]` |
| 3 | `3` (2) | `3` (2) | 2 | (9, 2) | thêm | `[[2,3],[6,1],[9,2]]` |

Ở Ví dụ 1: bước 1 sinh `(6, 3)`, bước 2 sinh `(6, 3)` ⇒ **gộp** thành `[[6,6]]`.

## 9. Code nộp LeetCode

Bản chọn nộp: **Cách 2** — hai con trỏ với lượng còn lại, không sửa input.

In [11]:
class Solution:
    def findRLEArray(self, encoded1: List[List[int]], encoded2: List[List[int]]) -> List[List[int]]:
        res = []
        i = j = 0
        r1, r2 = encoded1[0][1], encoded2[0][1]
        n1, n2 = len(encoded1), len(encoded2)
        while i < n1:
            f = r1 if r1 < r2 else r2
            v = encoded1[i][0] * encoded2[j][0]
            if res and res[-1][0] == v:
                res[-1][1] += f
            else:
                res.append([v, f])
            r1 -= f
            r2 -= f
            if r1 == 0:
                i += 1
                if i < n1:
                    r1 = encoded1[i][1]
            if r2 == 0:
                j += 1
                if j < n2:
                    r2 = encoded2[j][1]
        return res

for (e1, e2), exp in examples:
    got = Solution().findRLEArray(e1, e2); print(got, exp); assert got == exp

[[6, 6]] [[6, 6]]
[[2, 3], [6, 1], [9, 2]] [[2, 3], [6, 1], [9, 2]]


## 10. Vì sao thuật toán đúng

Gọi `p` = tổng các `f` đã sinh (số vị trí đã xử lý).

**Invariant:** trước mỗi vòng, (a) vị trí `p` nằm trong đoạn `i` của `nums1`, còn đúng `r1` phần tử của đoạn đó từ `p` trở đi; tương tự `j`, `r2` cho `nums2`; (b) `res` là RLE tối thiểu của `prodNums[0 .. p)`.

- *Khởi tạo:* `p = 0`, đầu đoạn 0 của cả hai, `res` rỗng ✓.
- *Một bước:* trên `[p, p + f)` với `f = min(r1, r2)`, cả hai mảng đều nằm trong đoạn hiện tại ⇒ `prodNums` hằng bằng `v1·v2` trên `f` vị trí đó. Nối `f` bản sao `v1·v2` vào RLE tối thiểu: nếu đoạn cuối cùng giá trị thì kéo dài, ngược lại thêm đoạn mới — kết quả vẫn tối thiểu (không tạo hai đoạn kề cùng giá trị). Sau khi trừ, đoạn nào còn 0 phần tử thì `p + f` là đầu đoạn kế tiếp ⇒ tiến con trỏ ⇒ (a) giữ.
- *Kết thúc:* mỗi bước `p` tăng `≥ 1`, và `i` hết khi `p = N`; vì hai mảng dài bằng nhau, `j` cũng hết đúng lúc đó. Khi đó `res` là RLE tối thiểu của toàn bộ `prodNums`. ∎

### 10.1 Kiểm chứng thực nghiệm số mảnh `≤ n₁ + n₂ − 1`

Đếm số bước của vòng lặp (số mảnh trước khi gộp) và so với cận `n₁ + n₂ − 1`; đồng thời kiểm tra kết quả **không** có hai đoạn kề cùng giá trị và không có tần suất 0.

In [12]:
def count_pieces(e1, e2):
    p1 = set(accumulate(f for _, f in e1)); p2 = set(accumulate(f for _, f in e2))
    return len(p1 | p2)

def rand_rle(rng, total, vmax, fmax):
    out, left = [], total
    while left:
        f = min(left, rng.randint(1, fmax)); v = rng.randint(1, vmax)
        if out and out[-1][0] == v:            # giữ input ở dạng tối thiểu (như đề minh họa)
            v = v % vmax + 1
        out.append([v, f]); left -= f
    return out

rng = random.Random(1010)
for _ in range(500):
    N = rng.randint(1, 40)
    e1, e2 = rand_rle(rng, N, 4, 5), rand_rle(rng, N, 4, 5)
    k = count_pieces(e1, e2)
    assert k <= len(e1) + len(e2) - 1
    r = Solution().findRLEArray(e1, e2)
    assert all(r[t][0] != r[t + 1][0] for t in range(len(r) - 1)) and all(f > 0 for _, f in r)
    assert len(r) <= k
print("Số mảnh ≤ n1 + n2 − 1, kết quả tối thiểu, không có tần suất 0 (500 cặp)")

Số mảnh ≤ n1 + n2 − 1, kết quả tối thiểu, không có tần suất 0 (500 cặp)


## 11. Phân tích Time / Space Complexity

Gọi `n₁ = len(encoded1)`, `n₂ = len(encoded2)` (`≤ 10⁵`), `N = Σ freq` (`≤ 10⁹`).

| Cách | Time | Space phụ trợ |
|---|---|---|
| 1. Giải nén – nhân – nén | `O(N)` | `O(N)` |
| 2. Hai con trỏ | `O(n₁ + n₂)` | `O(1)` (+ kết quả) |
| 3. Ranh giới + `bisect` | `O((n₁+n₂) log(n₁+n₂))` | `O(n₁ + n₂)` |
| 4. Trộn ranh giới tuyến tính | `O(n₁ + n₂)` | `O(1)` (+ kết quả) |

In [13]:
rng = random.Random(1868)
def big_rle(rng, total, fmax=10 ** 4):
    # chia total thành các đoạn có tần suất 1..fmax, giá trị 1..10^4
    out, left = [], total
    while left:
        f = min(left, rng.randint(1, fmax)); out.append([rng.randint(1, 10 ** 4), f]); left -= f
    return out
E1 = [[rng.randint(1, 10 ** 4), rng.randint(1, 10 ** 4)] for _ in range(10 ** 5)]
N = sum(f for _, f in E1)
E2 = big_rle(rng, N)
assert max(f for _, f in E1 + E2) <= 10 ** 4 and len(E2) <= 10 ** 5 + 5000
for name, f in [("two_pointers", rle_product_two_pointers), ("boundaries", rle_product_boundaries), ("submit", Solution().findRLEArray)]:
    t = time.perf_counter(); r = f(E1, E2); print(f"{name:<13}: {len(r)} đoạn, {(time.perf_counter() - t) * 1000:.0f} ms (n1 = {len(E1)}, n2 = {len(E2)}, N = {N:,})")

two_pointers : 199621 đoạn, 65 ms (n1 = 100000, n2 = 99656, N = 497,930,341)
boundaries   : 199621 đoạn, 201 ms (n1 = 100000, n2 = 99656, N = 497,930,341)


submit       : 199621 đoạn, 90 ms (n1 = 100000, n2 = 99656, N = 497,930,341)


## 12. Bộ test

In [14]:
ALL = {"two_pointers": rle_product_two_pointers, "boundaries": rle_product_boundaries, "submit": Solution().findRLEArray}
def run_all(e1, e2, brute=True):
    s1, s2 = [x[:] for x in e1], [x[:] for x in e2]
    res = {k: f(e1, e2) for k, f in ALL.items()}
    assert e1 == s1 and e2 == s2, "input bị sửa"
    if brute:
        res["brute"] = rle_product_bruteforce(e1, e2)
    vals = list(res.values())
    assert all(v == vals[0] for v in vals), (e1, e2, res)
    return vals[0]

for (e1, e2), exp in examples:
    assert run_all(e1, e2) == exp
# ví dụ tự dựng (đối chiếu lại đề gốc) — kỳ vọng kiểm bằng brute force trong run_all
edge = [
    (([[1, 1]], [[5, 1]]), [[5, 1]]),
    (([[1, 5]], [[1, 2], [2, 3]]), [[1, 2], [2, 3]]),
    (([[2, 2], [3, 2]], [[3, 2], [2, 2]]), [[6, 4]]),
    (([[10 ** 4, 10 ** 4]], [[10 ** 4, 10 ** 4]]), [[10 ** 8, 10 ** 4]]),
    (([[1, 1], [2, 1], [1, 1]], [[2, 1], [1, 1], [2, 1]]), [[2, 3]]),
    (([[1, 2], [1, 3]], [[4, 5]]), [[4, 5]]),
]
for (e1, e2), exp in edge:
    assert run_all(e1, e2) == exp, (e1, e2, run_all(e1, e2))
rng = random.Random(18680)
for _ in range(800):
    N = rng.randint(1, 60)
    run_all(rand_rle(rng, N, rng.choice([2, 3, 6]), rng.randint(1, 8)), rand_rle(rng, N, rng.choice([2, 3, 6]), rng.randint(1, 8)))
for _ in range(3):
    N = 10 ** 6
    a, b = big_rle(rng, N, 300), big_rle(rng, N, 400)
    assert len({tuple(map(tuple, f(a, b))) for f in ALL.values()}) == 1
print("Ví dụ, 6 edge case, 800 cặp nhỏ (đối chiếu brute), 3 cặp lớn: OK")

Ví dụ, 6 edge case, 800 cặp nhỏ (đối chiếu brute), 3 cặp lớn: OK


### 12.1 Kiểm tra tính chất

- Tổng tần suất của kết quả = `N` (độ dài mảng).
- Giao hoán: `findRLEArray(e1, e2) == findRLEArray(e2, e1)`.
- Nhân với mảng toàn `1` (`[[1, N]]`) cho lại `e1` ở dạng tối thiểu.
- Giải nén kết quả = tích từng phần tử của hai mảng giải nén.

In [15]:
rng = random.Random(121)
f = Solution().findRLEArray
for _ in range(500):
    N = rng.randint(1, 50)
    a, b = rand_rle(rng, N, 5, 6), rand_rle(rng, N, 5, 6)
    r = f(a, b)
    assert sum(x for _, x in r) == N
    assert r == f(b, a)
    assert f(a, [[1, N]]) == encode(decode(a))
    assert decode(r) == [x * y for x, y in zip(decode(a), decode(b))]
print("4 tính chất đúng trên 500 cặp")

4 tính chất đúng trên 500 cặp


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên gộp đoạn kề cùng giá trị

In [16]:
def wrong_no_merge(encoded1, encoded2):
    res = []; i = j = 0; r1, r2 = encoded1[0][1], encoded2[0][1]
    while i < len(encoded1):
        f = min(r1, r2)
        res.append([encoded1[i][0] * encoded2[j][0], f])            # SAI: luôn thêm mới
        r1 -= f; r2 -= f
        if r1 == 0:
            i += 1; r1 = encoded1[i][1] if i < len(encoded1) else 0
        if r2 == 0:
            j += 1; r2 = encoded2[j][1] if j < len(encoded2) else 0
    return res

(e1, e2), exp = examples[0]
print("sai :", wrong_no_merge(e1, e2), "| đúng:", exp)

sai : [[6, 3], [6, 3]] | đúng: [[6, 6]]


**Vì sao sai:** đề yêu cầu RLE **độ dài nhỏ nhất**; hai mảnh `1·6` và `2·3` cùng bằng 6 phải gộp. **Sửa:** `append_merge`.

### Lỗi 2 — Giả định các đoạn thẳng hàng (tiến cả hai con trỏ mỗi bước)

In [17]:
def wrong_aligned(encoded1, encoded2):
    res = []
    for (v1, f1), (v2, f2) in zip(encoded1, encoded2):               # SAI: đoạn không thẳng hàng
        append_merge(res, v1 * v2, min(f1, f2))
    return res

(e1, e2), exp = examples[1]
print("sai :", wrong_aligned(e1, e2), "| đúng:", exp)

sai : [[2, 3], [6, 1]] | đúng: [[2, 3], [6, 1], [9, 2]]


**Vì sao sai:** đoạn của hai mảng có ranh giới khác nhau; `zip` ghép đoạn thứ `k` với đoạn thứ `k` bất kể vị trí thật. **Sửa:** tiến con trỏ theo lượng còn lại.

### Lỗi 3 — Trừ trực tiếp vào `encoded[i][1]` (sửa input)

In [18]:
def mutating(encoded1, encoded2):
    res = []; i = j = 0
    while i < len(encoded1):
        f = min(encoded1[i][1], encoded2[j][1])
        append_merge(res, encoded1[i][0] * encoded2[j][0], f)
        encoded1[i][1] -= f; encoded2[j][1] -= f                     # hiệu ứng phụ
        if encoded1[i][1] == 0: i += 1
        if encoded2[j][1] == 0: j += 1
    return res

(e1, e2), exp = examples[1]
c1, c2 = [x[:] for x in e1], [x[:] for x in e2]
print("lần 1:", mutating(c1, c2), "| input sau khi gọi:", c1, c2)
try:
    print("lần 2:", mutating(c1, c2))
except IndexError as ex:
    print("lần 2 lỗi IndexError:", ex)

lần 1: [[2, 3], [6, 1], [9, 2]] | input sau khi gọi: [[1, 0], [2, 0], [3, 0]] [[2, 0], [3, 0]]
lần 2 lỗi IndexError: list index out of range


**Vì sao không tốt:** kết quả lần đầu đúng (và nhiều lời giải LeetCode làm vậy), nhưng input bị phá hủy — gọi lại hoặc dùng lại dữ liệu sẽ sai/crash. Trong phỏng vấn nên nêu rõ hoặc tránh. **Sửa:** biến `r1`, `r2` riêng.

### Lỗi 4 — Giải nén mảng lớn

In [19]:
for N in (10 ** 5, 10 ** 6, 2 * 10 ** 6):
    a = [[2, N // 2], [3, N - N // 2]]; b = [[5, N]]
    t = time.perf_counter(); rle_product_bruteforce(a, b); tb = time.perf_counter() - t
    t = time.perf_counter(); Solution().findRLEArray(a, b); ts = time.perf_counter() - t
    print(f"N={N:>9,}: giải nén {tb * 1000:7.0f} ms | hai con trỏ {ts * 1000:.3f} ms")

N=  100,000: giải nén       3 ms | hai con trỏ 0.007 ms
N=1,000,000: giải nén      42 ms | hai con trỏ 0.008 ms


N=2,000,000: giải nén      79 ms | hai con trỏ 0.007 ms


**Vì sao không tốt:** thời gian và bộ nhớ tỉ lệ với `N`; ở `N = 10⁹` là không thể. Hai con trỏ chỉ phụ thuộc **số đoạn**. **Sửa:** Cách 2.

### Lỗi 5 — Khi `r1 == r2`, chỉ tiến một con trỏ

In [20]:
def wrong_one_advance(encoded1, encoded2):
    res = []; i = j = 0; r1, r2 = encoded1[0][1], encoded2[0][1]
    while i < len(encoded1) and j < len(encoded2):
        f = min(r1, r2)
        res.append([encoded1[i][0] * encoded2[j][0], f])              # (không gộp để thấy rõ)
        r1 -= f; r2 -= f
        if r1 == 0:
            i += 1; r1 = encoded1[i][1] if i < len(encoded1) else 0
        elif r2 == 0:                                                 # SAI: 'elif'
            j += 1; r2 = encoded2[j][1] if j < len(encoded2) else 0
    return res

(e1, e2), exp = examples[0]
print("sai :", wrong_one_advance(e1, e2), "| đúng (chưa gộp):", wrong_no_merge(e1, e2))

sai : [[6, 3], [12, 0], [6, 3]] | đúng (chưa gộp): [[6, 3], [6, 3]]


**Vì sao sai:** khi cả hai đoạn cùng hết, dùng `elif` khiến `j` không tiến; vòng sau `r2 = 0` ⇒ sinh mảnh tần suất 0 với giá trị sai (`2·6 = 12`). **Sửa:** hai lệnh `if` độc lập.

### Lỗi 6 — Thêm điều kiện thừa khi gộp (đòi cả tần suất bằng nhau)

In [21]:
def wrong_merge_prev_piece(encoded1, encoded2):
    res = []; prev_v = None; i = j = 0; r1, r2 = encoded1[0][1], encoded2[0][1]
    while i < len(encoded1):
        f = min(r1, r2); v = encoded1[i][0] * encoded2[j][0]
        if v == prev_v and res[-1][1] == f:                            # SAI: điều kiện gộp thừa
            res[-1][1] += f
        else:
            res.append([v, f])
        prev_v = v
        r1 -= f; r2 -= f
        if r1 == 0:
            i += 1; r1 = encoded1[i][1] if i < len(encoded1) else 0
        if r2 == 0:
            j += 1; r2 = encoded2[j][1] if j < len(encoded2) else 0
    return res

e1, e2 = [[2, 2], [3, 1]], [[3, 2], [2, 1]]       # ví dụ tự dựng (đối chiếu lại đề gốc)
print("sai :", wrong_merge_prev_piece(e1, e2), "| đúng:", Solution().findRLEArray(e1, e2), "| brute:", rle_product_bruteforce(e1, e2))

sai : [[6, 2], [6, 1]] | đúng: [[6, 3]] | brute: [[6, 3]]


**Vì sao sai:** điều kiện gộp chỉ cần **giá trị** bằng nhau; thêm điều kiện về tần suất (hay bất kỳ điều kiện nào khác) làm sót trường hợp. **Sửa:** `if res and res[-1][0] == v: res[-1][1] += f`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Dữ liệu nén (RLE, khoảng, danh sách đoạn) ⇒ **đừng giải nén**; làm việc trên đoạn bằng hai con trỏ.
- "Tiêu thụ `min(còn lại)`" là khuôn chung cho mọi phép toán từng phần tử trên hai chuỗi đoạn (cộng, nhân, so sánh, AND/OR khoảng).
- Kết quả "tối thiểu" ⇒ gộp vào phần tử cuối khi trùng khóa.
- Hai lệnh `if` độc lập cho hai con trỏ — chúng có thể tiến **cùng lúc**.
- Liên quan: **986 Interval List Intersections**, **1570 Dot Product of Two Sparse Vectors**, **900 RLE Iterator**.

## 15. Biến thể của bài

1. **Tổng / hiệu / max** từng phần tử của hai mảng RLE — chỉ đổi phép `v1·v2` (Bài tập 4).
2. **Tích vô hướng** (dot product) của hai mảng RLE — không cần kết quả RLE, cộng dồn `v1·v2·f`.
3. **Tích của `k` mảng RLE** — `k` con trỏ, mỗi bước `f = min` các lượng còn lại (Bài tập 5).
4. **900 RLE Iterator**: lấy phần tử thứ `n` tiếp theo mà không giải nén.
5. **986 Interval List Intersections**: cùng khuôn hai con trỏ trên danh sách khoảng.

### 15.1 Code biến thể: tích vô hướng và RLE Iterator

In [22]:
def rle_dot(e1, e2):
    total, i, j = 0, 0, 0
    r1, r2 = e1[0][1], e2[0][1]
    while i < len(e1):
        f = min(r1, r2); total += e1[i][0] * e2[j][0] * f
        r1 -= f; r2 -= f
        if r1 == 0:
            i += 1; r1 = e1[i][1] if i < len(e1) else 0
        if r2 == 0:
            j += 1; r2 = e2[j][1] if j < len(e2) else 0
    return total

class RLEIterator:                                     # 900 (định dạng [val, freq])
    def __init__(self, enc):
        self.enc, self.i, self.used = enc, 0, 0
    def next(self, n):
        while self.i < len(self.enc):
            v, f = self.enc[self.i]
            if self.used + n <= f:
                self.used += n; return v
            n -= f - self.used; self.i += 1; self.used = 0
        return -1

rng = random.Random(15)
for _ in range(300):
    N = rng.randint(1, 30); a, b = rand_rle(rng, N, 5, 5), rand_rle(rng, N, 5, 5)
    assert rle_dot(a, b) == sum(x * y for x, y in zip(decode(a), decode(b)))
    it, flat, pos = RLEIterator(a), decode(a), 0
    for _ in range(5):
        k = rng.randint(1, 8); got = it.next(k); pos += k
        assert got == (flat[pos - 1] if pos <= len(flat) else -1)
print("rle_dot và RLEIterator khớp vét cạn;", rle_dot(*examples[1][0]), "=", sum(decode([[2, 3], [6, 1], [9, 2]])))

rle_dot và RLEIterator khớp vét cạn; 30 = 30


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Giải nén | Nghĩa đen của đề | `O(N)` | `O(N)` | Kiểm chứng |
| 2. Hai con trỏ | Tiêu thụ `min(còn lại)` | `O(n₁+n₂)` | `O(1)` | **Bản nộp** |
| 3. Ranh giới + `bisect` | Hợp ranh giới | `O(n log n)` | `O(n)` | Khi đã có tổng tiền tố |
| 4. Trộn ranh giới | Vị trí tuyệt đối | `O(n₁+n₂)` | `O(1)` | Cùng ý Cách 2 |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** (a) `[[1,2]] × [[3,2]]`; (b) `[[2,1],[3,1]] × [[3,1],[2,1]]`; (c) `[[1,1],[2,1],[1,1]] × [[2,1],[1,1],[2,1]]`; (d) `[[4,3]] × [[1,1],[2,2]]`.

**Bài 2 (tính tay):** Trace hai con trỏ cho `[[3,4],[1,2]] × [[2,1],[3,3],[1,2]]`, ghi từng mảnh và kết quả.

**Bài 3 (điền TODO):** Hoàn thiện `append_merge_todo(res, v, f)`.

**Bài 4 (tự viết lại):** Viết `rle_op(e1, e2, op)` tổng quát (`op` là hàm hai biến), dùng cho tổng và max; đối chiếu brute force.

**Bài 5 (mini challenge):** Viết `rle_product_k(encs)` nhân **`k`** mảng RLE cùng độ dài, `O(k · Σnᵢ)`; đối chiếu brute force.

In [23]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def append_merge_todo(res, v, f):
    # TODO: nếu res khác rỗng và res[-1][0] == v: cộng f vào res[-1][1]; ngược lại append [v, f]
    pass

In [24]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
def rle_op(e1, e2, op):
    # TODO: như Cách 2 nhưng giá trị mảnh là op(v1, v2)
    pass

def rle_product_k(encs):
    # TODO: con trỏ idx[t] và còn lại rem[t] cho mỗi mảng; f = min(rem); tích các giá trị hiện tại
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: tính từng mảnh. Mức 2: gộp mảnh cùng giá trị. Mức 3: (a) `[[3,2]]`; (b) `[[6,2]]`; (c) `[[2,3]]`; (d) `[[4,1],[8,2]]`.

**Bài 2.** Mức 1: mảnh 1: `3·2` dài 1. Mức 2: mảnh 2: `3·3` dài 3 (đoạn `3` của mảng 1 còn 3). Mức 3: mảnh 3: `1·1` dài 2 ⇒ `[[6,1],[9,3],[1,2]]`.

**Bài 3.** Mức 1: kiểm tra rỗng trước. Mức 2: so giá trị. Mức 3: sửa tại chỗ `res[-1][1]`.

**Bài 4.** Mức 1: thay phép nhân bằng `op`. Mức 2: vẫn gộp. Mức 3: thử `op = max` và `op = lambda a, b: a + b`.

**Bài 5.** Mức 1: `f = min(rem)`. Mức 2: trừ `f` khỏi mọi `rem[t]`, tiến các con trỏ có `rem[t] = 0`. Mức 3: dừng khi con trỏ đầu tiên hết (mọi mảng cùng độ dài).

## 19. Đáp án tham khảo cho bài tập

In [25]:
# Bài 1, 2, 3
f = Solution().findRLEArray
b1 = [f(a, b) for a, b in [([[1, 2]], [[3, 2]]), ([[2, 1], [3, 1]], [[3, 1], [2, 1]]),
                           ([[1, 1], [2, 1], [1, 1]], [[2, 1], [1, 1], [2, 1]]), ([[4, 3]], [[1, 1], [2, 2]])]]
print("Bài 1:", b1); assert b1 == [[[3, 2]], [[6, 2]], [[2, 3]], [[4, 1], [8, 2]]]
print("Bài 2:"); assert trace_two_pointers([[3, 4], [1, 2]], [[2, 1], [3, 3], [1, 2]]) == [[6, 1], [9, 3], [1, 2]]
def append_merge_todo(res, v, f):
    if res and res[-1][0] == v:
        res[-1][1] += f
    else:
        res.append([v, f])
r = []
for v, k in [(1, 2), (1, 3), (2, 1), (1, 1)]:
    append_merge_todo(r, v, k)
print("Bài 3:", r); assert r == [[1, 5], [2, 1], [1, 1]]

Bài 1: [[[3, 2]], [[6, 2]], [[2, 3]], [[4, 1], [8, 2]]]
Bài 2:
i=0 (v1=3, còn 4) | j=0 (v2=2, còn 1) -> mảnh (6, 1) thêm res=[[6, 1]]
i=0 (v1=3, còn 3) | j=1 (v2=3, còn 3) -> mảnh (9, 3) thêm res=[[6, 1], [9, 3]]
i=1 (v1=1, còn 2) | j=2 (v2=1, còn 2) -> mảnh (1, 2) thêm res=[[6, 1], [9, 3], [1, 2]]
Bài 3: [[1, 5], [2, 1], [1, 1]]


In [26]:
# Bài 4, 5
def rle_op(e1, e2, op):
    res = []; i = j = 0; r1, r2 = e1[0][1], e2[0][1]
    while i < len(e1):
        k = min(r1, r2); append_merge(res, op(e1[i][0], e2[j][0]), k)
        r1 -= k; r2 -= k
        if r1 == 0:
            i += 1; r1 = e1[i][1] if i < len(e1) else 0
        if r2 == 0:
            j += 1; r2 = e2[j][1] if j < len(e2) else 0
    return res

def rle_product_k(encs):
    k = len(encs); idx = [0] * k; rem = [e[0][1] for e in encs]; res = []
    while idx[0] < len(encs[0]):
        f = min(rem); v = 1
        for t in range(k):
            v *= encs[t][idx[t]][0]
        append_merge(res, v, f)
        for t in range(k):
            rem[t] -= f
            if rem[t] == 0:
                idx[t] += 1
                rem[t] = encs[t][idx[t]][1] if idx[t] < len(encs[t]) else 0
    return res

rng = random.Random(5)
for _ in range(300):
    N = rng.randint(1, 30)
    a, b = rand_rle(rng, N, 4, 5), rand_rle(rng, N, 4, 5)
    for op in (max, lambda x, y: x + y, lambda x, y: x * y):
        assert rle_op(a, b, op) == encode([op(x, y) for x, y in zip(decode(a), decode(b))])
    encs = [rand_rle(rng, N, 3, 5) for _ in range(rng.randint(1, 4))]
    prod = [1] * N
    for e in encs:
        prod = [p * x for p, x in zip(prod, decode(e))]
    assert rle_product_k(encs) == encode(prod)
print("Bài 4, 5: OK |", rle_op(*examples[1][0], lambda x, y: x + y), rle_product_k([*examples[1][0], [[2, 6]]]))

Bài 4, 5: OK | [[3, 3], [5, 1], [6, 2]] [[4, 3], [12, 1], [18, 2]]


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template (hai con trỏ trên RLE):**

```python
res, i, j = [], 0, 0
r1, r2 = e1[0][1], e2[0][1]
while i < len(e1):
    f = min(r1, r2); v = e1[i][0] * e2[j][0]
    if res and res[-1][0] == v: res[-1][1] += f     # gộp
    else: res.append([v, f])
    r1 -= f; r2 -= f
    if r1 == 0: i += 1; r1 = e1[i][1] if i < len(e1) else 0
    if r2 == 0: j += 1; r2 = e2[j][1] if j < len(e2) else 0   # 'if', không phải 'elif'
return res
```

**Checklist:**
- [ ] Không giải nén (độ dài thật tới `10⁹`)?
- [ ] Tiêu thụ `min(r1, r2)` và tiến **cả hai** con trỏ khi cùng hết?
- [ ] Gộp vào đoạn cuối khi cùng giá trị (kết quả tối thiểu)?
- [ ] Không có tần suất 0 trong kết quả?
- [ ] Không sửa input (hoặc nêu rõ nếu có)?